In [184]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch
import torch.optim as optim

In [ ]:
# Load the dataset
data = pd.read_csv('winequality-red.csv')



In [186]:
# Prepare the data

X = data.drop('quality', axis=1).values
y = data['quality'].values.reshape(-1, 1)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

scaled = StandardScaler()
X_train = scaled.fit_transform(X_train)
X_test = scaled.transform(X_test)


X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


print(X_train.shape, y_train.shape)


torch.Size([1071, 11]) torch.Size([1071, 1])


In [187]:
# Define the neural network model and training loop

class WineQualityModel(nn.Module):
    def __init__(self):
        super(WineQualityModel, self).__init__()
        self.fc1 = nn.Linear(11, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x) 
        return x
model = WineQualityModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/200], Loss: 6.2856
Epoch [20/200], Loss: 3.1537
Epoch [30/200], Loss: 1.4925
Epoch [40/200], Loss: 1.0416
Epoch [50/200], Loss: 0.9053
Epoch [60/200], Loss: 0.7185
Epoch [70/200], Loss: 0.6115
Epoch [80/200], Loss: 0.5363
Epoch [90/200], Loss: 0.4814
Epoch [100/200], Loss: 0.4402
Epoch [110/200], Loss: 0.4094
Epoch [120/200], Loss: 0.3859
Epoch [130/200], Loss: 0.3681
Epoch [140/200], Loss: 0.3546
Epoch [150/200], Loss: 0.3443
Epoch [160/200], Loss: 0.3352
Epoch [170/200], Loss: 0.3265
Epoch [180/200], Loss: 0.3178
Epoch [190/200], Loss: 0.3109
Epoch [200/200], Loss: 0.3051


In [188]:
# Test the model


model.eval()

with torch.no_grad():
    test_outputs = model(X_test)
    test_loss = criterion(test_outputs, torch.tensor(y_test, dtype=torch.float32))
    print(f'Test Loss: {test_loss.item():.4f}')
    print(f"Mean Error: {torch.sqrt(test_loss).item():.4f}")
    

for i in range(10):
    pred = test_outputs[i].item()
    real = y_test[i].item()
    print(f" bottle {i+1}: Predicted {pred:.2f} | Actual {real}")

Test Loss: 0.4067
Mean Error: 0.6377
 bottle 1: Predicted 5.27 | Actual 6.0
 bottle 2: Predicted 5.13 | Actual 5.0
 bottle 3: Predicted 5.37 | Actual 6.0
 bottle 4: Predicted 5.48 | Actual 5.0
 bottle 5: Predicted 5.90 | Actual 6.0
 bottle 6: Predicted 5.25 | Actual 5.0
 bottle 7: Predicted 5.30 | Actual 5.0
 bottle 8: Predicted 4.77 | Actual 5.0
 bottle 9: Predicted 6.03 | Actual 5.0
 bottle 10: Predicted 5.94 | Actual 6.0


C:\Users\Dima\AppData\Local\Temp\ipykernel_26876\2845436062.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_loss = criterion(test_outputs, torch.tensor(y_test, dtype=torch.float32))
